# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adityag30/FlyRank-internship-ML/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



### Finding 1: The model outperformed the rule-based baseline.

The paper reports that the machine learning model achieved better ranking performance than the rule-based baseline.

**Methodology question:**  
Where does the prediction label come from? If the label is derived from the same observation window as the input features, the model may be learning current conditions rather than predicting future outcomes. A future-window label would provide stronger evidence that the model can generalize to new data.



### Finding 2: The evaluation shows strong model performance.

The paper reports improved performance using evaluation metrics such as Precision@K.

**Methodology question:**  
Does the validation design support this claim? If related pages or pages from the same client appear in both the training and testing sets, the reported performance may be optimistic. A grouped-by-client or time-aware validation would provide stronger evidence that the model generalizes to unseen data.


These questions are intended to strengthen the evidence presented in the paper rather than criticize it. They focus on understanding how labels were defined and whether the validation strategy supports the reported conclusions.

In [1]:
pass

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


The Week 5 model was originally evaluated using a random train-test split. I attempted to repeat the evaluation using a grouped split based on client identifiers, but the extracted dataset did not include a `client_hash_id` field. Therefore, a grouped validation could not be performed.

The random split achieved an accuracy of 1.00. However, this result should be interpreted cautiously because the target label was derived from the same observable features used by the model. This likely inflates the measured performance and does not necessarily reflect how the model would perform on unseen future data.

In [4]:
from google.colab import userdata
import duckdb
import pandas as pd

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(
    f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

REL = "hf://datasets/FlyRank/internship-warehouse"

query = f"""
SELECT
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions,
    SUM(gsc_clicks) AS gsc_clicks,
    AVG(gsc_avg_position) AS gsc_avg_position,
    SUM(ga4_sessions) AS ga4_sessions,
    SUM(ga4_engaged_sessions) AS ga4_engaged_sessions
FROM read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')
WHERE month='2026-03'
GROUP BY content_hash_id
"""

df = con.sql(query).df()

df["ctr"] = df["gsc_clicks"] / df["gsc_impressions"].replace(0, 1)
df["engagement_rate"] = (
    df["ga4_engaged_sessions"] /
    df["ga4_sessions"].replace(0, 1)
)

feature_cols = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_avg_position",
    "ga4_sessions",
    "ga4_engaged_sessions",
    "ctr",
    "engagement_rate"
]

X = df[feature_cols]

y = (
    (df["gsc_impressions"] >= 10000) &
    (df["ctr"] < 0.03)
).astype(int)

print(df.shape)
df.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

(331437, 8)


,content_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ga4_sessions,ga4_engaged_sessions,ctr,engagement_rate
0,content_b7e512995f79d5a6,1140.0,2.0,4.394234,0.0,0.0,0.001754,0.0
1,content_05597932fe4da067,57.0,0.0,2.714744,0.0,0.0,0.000000,0.0
2,content_905aa32a0230694e,149.0,0.0,6.481453,4.0,0.0,0.000000,0.0
3,content_05434271b257bb68,1421.0,6.0,6.320337,9.0,0.0,0.004222,0.0
4,content_d056587ff7faca0c,2770.0,16.0,4.459107,3.0,0.0,0.005776,0.0


In [5]:
# This cell is for CODE (numbers, a query, a check).
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import accuracy_score
import pandas as pd

# -----------------------------
# Random Split (Week 5)
# -----------------------------
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

model_random = DecisionTreeClassifier(max_depth=3, random_state=42)
model_random.fit(X_train_r, y_train_r)

random_acc = accuracy_score(
    y_test_r,
    model_random.predict(X_test_r)
)

# -----------------------------
# Honest Split (Grouped if available)
# -----------------------------
if "client_hash_id" in df.columns:

    groups = df["client_hash_id"]

    splitter = GroupShuffleSplit(
        test_size=0.20,
        random_state=42,
        n_splits=1
    )

    train_idx, test_idx = next(splitter.split(X, y, groups))

    X_train_g = X.iloc[train_idx]
    X_test_g = X.iloc[test_idx]

    y_train_g = y.iloc[train_idx]
    y_test_g = y.iloc[test_idx]

    model_group = DecisionTreeClassifier(max_depth=3, random_state=42)
    model_group.fit(X_train_g, y_train_g)

    group_acc = accuracy_score(
        y_test_g,
        model_group.predict(X_test_g)
    )

    results = pd.DataFrame({
        "Validation": ["Random Split", "Grouped Split"],
        "Accuracy": [random_acc, group_acc]
    })

else:

    results = pd.DataFrame({
        "Validation": ["Random Split"],
        "Accuracy": [random_acc]
    })

    print("client_hash_id not available. Grouped validation could not be performed.")

display(results)

client_hash_id not available. Grouped validation could not be performed.


,Validation,Accuracy
0,Random Split,1.0


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*



The final feature set was reviewed for information that would not be available at prediction time. During this audit, I found that the target label was created using `gsc_impressions` and `ctr`, while these same variables were also included as model features. This creates label leakage because the model can directly learn the rule used to generate the labels instead of learning an independent relationship.

No future metrics or product-generated scores were used as features. However, the overlap between the target definition and the input features likely explains the perfect accuracy observed in Week 5. Therefore, the reported performance should be interpreted as an upper bound rather than evidence of real-world predictive ability.

In [6]:
# This cell is for CODE (numbers, a query, a check).
import pandas as pd

leakage_audit = pd.DataFrame({
    "Feature": feature_cols,
    "Used in Model": ["Yes"] * len(feature_cols),
    "Used to Create Label": [
        "Yes",   # gsc_impressions
        "No",    # gsc_clicks
        "No",    # gsc_avg_position
        "No",    # ga4_sessions
        "No",    # ga4_engaged_sessions
        "Yes",   # ctr
        "No"     # engagement_rate
    ],
    "Leakage Risk": [
        "High",
        "Low",
        "Low",
        "Low",
        "Low",
        "High",
        "Low"
    ]
})

display(leakage_audit)

,Feature,Used in Model,Used to Create Label,Leakage Risk
0,gsc_impressions,Yes,Yes,High
1,gsc_clicks,Yes,No,Low
2,gsc_avg_position,Yes,No,Low
3,ga4_sessions,Yes,No,Low
4,ga4_engaged_sessions,Yes,No,Low
5,ctr,Yes,Yes,High
6,engagement_rate,Yes,No,Low


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*



### Original claim

The Decision Tree model accurately identifies pages that should be refreshed.

### Revised claim

On the March 2026 evaluation dataset, the Decision Tree reproduced the baseline rule with high measured accuracy. However, this result is influenced by label leakage because the target was derived from features also used by the model. Therefore, the model should be viewed as a **decision-support tool** that demonstrates observed patterns in this dataset rather than as evidence that it will accurately identify refresh opportunities on unseen future data.

In [7]:
# This cell is for CODE (numbers, a query, a check).
pass

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.